In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report 


In [2]:
import os
print(os.getcwd())


C:\Users\User\Downloads\bank cred scroe project


In [3]:
cust_enq = pd.read_csv("Cust_Enquiry.csv")
cust_acc = pd.read_csv("Cust_Account.csv")
cust_demo = pd.read_csv("Cust_Demographics.csv")

C:\Users\User\AppData\Local\Temp\ipykernel_17196\2258738253.py:2: DtypeWarning: Columns (0: rateofinterest) have mixed types. Specify dtype option on import or set low_memory=False.
  cust_acc = pd.read_csv("Cust_Account.csv")
C:\Users\User\AppData\Local\Temp\ipykernel_17196\2258738253.py:3: DtypeWarning: Columns (0: feature_10, 1: feature_18, 2: feature_61) have mixed types. Specify dtype option on import or set low_memory=False.
  cust_demo = pd.read_csv("Cust_Demographics.csv")


In [4]:
cust_acc.head()

,dt_opened,customer_no,upload_dt,acct_type,owner_indic,opened_dt,last_paymt_dt,closed_dt,reporting_dt,high_credit_amt,...,amt_past_due,paymenthistory1,paymenthistory2,paymt_str_dt,paymt_end_dt,creditlimit,cashlimit,rateofinterest,paymentfrequency,actualpaymentamount
0,10-Nov-15,12265,20-Oct-15,6,1,09-Jun-13,30-Jun-14,05-Jul-14,30-Sep-15,20900.0,...,NaN,"""""""STDSTDSTDXXXXXXXXXXXXXXXSTDXXXXXXXXXXXXXXXS...",NaN,01-Sep-15,01-Jul-14,NaN,NaN,NaN,NaN,NaN
1,10-Nov-15,12265,20-Oct-15,10,1,25-May-12,06-Sep-15,NaN,03-Oct-15,16201.0,...,NaN,"""""""0000000000000000000000000000000000000000000...","""""""000000000000000000000000000XXX0000000000000...",01-Oct-15,01-Nov-12,14000.0,1400.0,NaN,3.0,5603.0
2,10-Nov-15,12265,20-Oct-15,10,1,22-Mar-12,31-Aug-15,NaN,30-Sep-15,41028.0,...,NaN,"""""""0000000000000000000000000000000000000000000...","""""""0000000000000000000000000000000000000000000...",01-Sep-15,01-Oct-12,NaN,NaN,NaN,NaN,NaN
3,20-Jul-15,15606,09-Jul-15,10,1,13-Jan-06,NaN,26-Jul-07,31-Jan-09,93473.0,...,NaN,"""""""1200900600600600300000000000000000000000000...",NaN,01-Jul-07,01-Feb-06,NaN,NaN,NaN,NaN,NaN
4,20-Jul-15,15606,09-Jul-15,6,1,18-Jan-15,05-May-15,NaN,31-May-15,20250.0,...,NaN,"""""""000000000000000""""""",NaN,01-May-15,01-Jan-15,NaN,NaN,NaN,NaN,NaN


In [5]:
cust_demo.head()

,dt_opened,customer_no,entry_time,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,...,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,feature_79,Bad_label
0,18-Apr-15,1,13-Apr-15,Insignia,13-Apr-15,650.0,2.0,Card Setup,14.0,500000.0,...,21.0,R,NaN,NaN,0000-00-00,0.0,98332XXXXX,1.0,N,0
1,21-Apr-15,2,21-Apr-15,Insignia,21-Apr-15,760.0,1.0,Card Setup,14.0,1200000.0,...,17.0,R,NaN,NaN,0000-00-00,0.0,99455XXXXX,1.0,N,0
2,22-Apr-15,3,21-Apr-15,Insignia,21-Apr-15,774.0,1.0,Card Setup,14.0,700000.0,...,17.0,R,NaN,NaN,0000-00-00,0.0,98456XXXXX,1.0,N,0
3,25-Apr-15,4,15-Apr-15,Insignia,20-Apr-15,770.0,1.0,Card Setup,14.0,500000.0,...,21.0,R,NaN,NaN,6/15/65,1.0,98220XXXXX,1.0,N,0
4,06-May-15,5,30-Apr-15,Insignia,NaN,NaN,3.0,Card Setup,14.0,500000.0,...,13.0,R,NaN,NaN,0000-00-00,0.0,98111XXXXX,1.0,N,0


In [6]:
print("Cust_Acc =", cust_acc.shape)
print("Cust_Enq =", cust_enq.shape)
print("Cust_Demo =", cust_demo.shape)

Cust_Acc = (186329, 21)
Cust_Enq = (413188, 6)
Cust_Demo = (23896, 83)


In [7]:
cust_acc.columns

Index(['dt_opened', 'customer_no', 'upload_dt', 'acct_type', 'owner_indic',
       'opened_dt', 'last_paymt_dt', 'closed_dt', 'reporting_dt',
       'high_credit_amt', 'cur_balance_amt', 'amt_past_due', 'paymenthistory1',
       'paymenthistory2', 'paymt_str_dt', 'paymt_end_dt', 'creditlimit',
       'cashlimit', 'rateofinterest', 'paymentfrequency',
       'actualpaymentamount'],
      dtype='str')

In [8]:
data_cols_acc = ["dt_opened","upload_dt", "opened_dt","last_paymt_dt","closed_dt","reporting_dt",
                 "paymt_str_dt","paymt_end_dt"]

In [9]:
for col in data_cols_acc:
    cust_acc[col] = pd.to_datetime(cust_acc[col], errors = "coerce", dayfirst = True)

C:\Users\User\AppData\Local\Temp\ipykernel_17196\3097195435.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cust_acc[col] = pd.to_datetime(cust_acc[col], errors = "coerce", dayfirst = True)
C:\Users\User\AppData\Local\Temp\ipykernel_17196\3097195435.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cust_acc[col] = pd.to_datetime(cust_acc[col], errors = "coerce", dayfirst = True)
C:\Users\User\AppData\Local\Temp\ipykernel_17196\3097195435.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cust_acc[col] = pd.to_datetime(cust_acc[col], errors = "coerce", dayfirst = True

In [10]:
cust_enq.columns

Index(['dt_opened', 'customer_no', 'upload_dt', 'enquiry_dt', 'enq_purpose',
       'enq_amt'],
      dtype='str')

In [11]:
date_cols_enq = ["dt_opened","upload_dt","enquiry_dt"]

for col in date_cols_enq:
    cust_enq[col] = pd.to_datetime(cust_enq[col], errors= "coerce", dayfirst = True)

C:\Users\User\AppData\Local\Temp\ipykernel_17196\3931608539.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cust_enq[col] = pd.to_datetime(cust_enq[col], errors= "coerce", dayfirst = True)
C:\Users\User\AppData\Local\Temp\ipykernel_17196\3931608539.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cust_enq[col] = pd.to_datetime(cust_enq[col], errors= "coerce", dayfirst = True)
C:\Users\User\AppData\Local\Temp\ipykernel_17196\3931608539.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cust_enq[col] = pd.to_datetime(cust_enq[col], errors= "coerce", dayfirst = True)


In [12]:
cust_demo.columns

Index(['dt_opened', 'customer_no', 'entry_time', 'feature_1', 'feature_2',
       'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7',
       'feature_8', 'feature_9', 'feature_10', 'feature_11', 'feature_12',
       'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17',
       'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22',
       'feature_23', 'feature_24', 'feature_25', 'feature_26', 'feature_27',
       'feature_28', 'feature_29', 'feature_30', 'feature_31', 'feature_32',
       'feature_33', 'feature_34', 'feature_35', 'feature_36', 'feature_37',
       'feature_38', 'feature_39', 'feature_40', 'feature_41', 'feature_42',
       'feature_43', 'feature_44', 'feature_45', 'feature_46', 'feature_47',
       'feature_48', 'feature_49', 'feature_50', 'feature_51', 'feature_52',
       'feature_53', 'feature_54', 'feature_55', 'feature_56', 'feature_57',
       'feature_58', 'feature_59', 'feature_60', 'feature_61', 'feature_62',
       '

In [13]:
cust_acc.head()

,dt_opened,customer_no,upload_dt,acct_type,owner_indic,opened_dt,last_paymt_dt,closed_dt,reporting_dt,high_credit_amt,...,amt_past_due,paymenthistory1,paymenthistory2,paymt_str_dt,paymt_end_dt,creditlimit,cashlimit,rateofinterest,paymentfrequency,actualpaymentamount
0,2015-11-10,12265,2015-10-20,6,1,2013-06-09,2014-06-30,2014-07-05,2015-09-30,20900.0,...,NaN,"""""""STDSTDSTDXXXXXXXXXXXXXXXSTDXXXXXXXXXXXXXXXS...",NaN,2015-09-01,2014-07-01,NaN,NaN,NaN,NaN,NaN
1,2015-11-10,12265,2015-10-20,10,1,2012-05-25,2015-09-06,NaT,2015-10-03,16201.0,...,NaN,"""""""0000000000000000000000000000000000000000000...","""""""000000000000000000000000000XXX0000000000000...",2015-10-01,2012-11-01,14000.0,1400.0,NaN,3.0,5603.0
2,2015-11-10,12265,2015-10-20,10,1,2012-03-22,2015-08-31,NaT,2015-09-30,41028.0,...,NaN,"""""""0000000000000000000000000000000000000000000...","""""""0000000000000000000000000000000000000000000...",2015-09-01,2012-10-01,NaN,NaN,NaN,NaN,NaN
3,2015-07-20,15606,2015-07-09,10,1,2006-01-13,NaT,2007-07-26,2009-01-31,93473.0,...,NaN,"""""""1200900600600600300000000000000000000000000...",NaN,2007-07-01,2006-02-01,NaN,NaN,NaN,NaN,NaN
4,2015-07-20,15606,2015-07-09,6,1,2015-01-18,2015-05-05,NaT,2015-05-31,20250.0,...,NaN,"""""""000000000000000""""""",NaN,2015-05-01,2015-01-01,NaN,NaN,NaN,NaN,NaN


In [19]:
print("CUSTOMER ACCOUNT")
cust_acc.isnull().sum().sort_values(ascending = False)


CUSTOMER ACCOUNT


amt_past_due           185453
rateofinterest         161496
cashlimit              151047
actualpaymentamount    145276
creditlimit            137477
paymentfrequency       122436
closed_dt              109075
paymenthistory2        107824
last_paymt_dt           25487
high_credit_amt          8875
opened_dt                 455
paymt_end_dt                1
paymt_str_dt                1
dt_opened                   0
paymenthistory1             0
customer_no                 0
reporting_dt                0
owner_indic                 0
acct_type                   0
upload_dt                   0
cur_balance_amt             0
dtype: int64

In [20]:
print("CUSTOMER ENQUIRY")
cust_enq.isnull().sum().sort_values(ascending = False)


CUSTOMER ENQUIRY


upload_dt      110
enquiry_dt     110
enq_purpose    110
enq_amt        110
dt_opened        0
customer_no      0
dtype: int64

In [21]:
print("CUSTOMER DEMOGRAPHICS")
cust_demo.isnull().sum().sort_values(ascending = False)

CUSTOMER DEMOGRAPHICS


feature_61     23887
feature_74     23879
feature_18     23878
feature_10     23845
feature_49     23792
               ...  
feature_47         0
customer_no        0
feature_22         0
feature_20         0
Bad_label          0
Length: 83, dtype: int64

In [22]:
print(cust_demo["Bad_label"].value_counts())

Bad_label
0    22892
1     1004
Name: count, dtype: int64


In [28]:
print("CUSTOMER ACCOUNT - > ", cust_acc["customer_no"].nunique())
print("CUSTOMER ENQUIRY - > ", cust_enq["customer_no"].nunique())
print("CUSTOMER DEMOGRAPHICS - > ", cust_demo["customer_no"].nunique())

CUSTOMER ACCOUNT - >  23896
CUSTOMER ENQUIRY - >  23896
CUSTOMER DEMOGRAPHICS - >  23896


In [30]:
num_cols_to_fix = [
    "high_credit_amt", "cur_balance_amt", "creditlimit", "cashlimit",
    "amt_past_due", "rateofinterest", "actualpaymentamount"
]
for c in num_cols_to_fix:
    if c in cust_acc.columns:
        cust_acc[c] = pd.to_numeric(cust_acc[c], errors="coerce")

In [31]:
account = cust_acc.groupby("customer_no").agg(
        total_high_credit_amt=("high_credit_amt", "sum"),
    total_cur_balance_amt=("cur_balance_amt", "sum"),
    mean_cur_balance_amt=("cur_balance_amt", "mean"),
    total_credit_limit=("creditlimit", "sum"),
    mean_credit_limit=("creditlimit", "mean"),
    total_cash_limit=("cashlimit", "sum"),
    mean_cash_limit=("cashlimit", "mean"),
    total_amt_past_due=("amt_past_due", "sum"),
    num_accounts=("customer_no", "count"),
    mean_rateofinterest=("rateofinterest", "mean"),
).reset_index()

In [32]:
acc_num_cols = [
    "high_credit_amt", "cur_balance_amt", "creditlimit", "cashlimit",
    "amt_past_due", "rateofinterest", "actualpaymentamount"
]
for c in acc_num_cols:
    if c in cust_acc.columns:
        cust_acc[c] = pd.to_numeric(cust_acc[c], errors="coerce")
 
enq_num_cols = ["enq_amt"]
for c in enq_num_cols:
    if c in cust_enq.columns:
        cust_enq[c] = pd.to_numeric(cust_enq[c], errors="coerce")
 
demo_num_cols = [c for c in cust_demo.columns if c.startswith("feature_")]
for c in demo_num_cols:
    cust_demo[c] = pd.to_numeric(cust_demo[c], errors="coerce")

In [34]:
account["ratio_currbalance_creditlimit"] = (
    account["total_cur_balance_amt"] / account["total_credit_limit"].replace(0, np.nan)
)
 
cust_acc["diff_lastpaymt_opened_dt"] = (
    cust_acc["last_paymt_dt"] - cust_acc["opened_dt"]
).dt.days
 
time_agg = cust_acc.groupby("customer_no").agg(
    total_diff_lastpaymt_opened_dt=("diff_lastpaymt_opened_dt", "sum"),
    mean_diff_lastpaymt_opened_dt=("diff_lastpaymt_opened_dt", "mean"),
).reset_index()
 
account = account.merge(time_agg, on="customer_no", how="left")

In [35]:
cust_enq["diff_open_enquiry_dt"] = (
    cust_enq["enquiry_dt"] - cust_enq["dt_opened"]
).dt.days
 
max_enq_date = cust_enq["enquiry_dt"].max()
cust_enq["days_since_enquiry"] = (max_enq_date - cust_enq["enquiry_dt"]).dt.days
 
enq_agg = cust_enq.groupby("customer_no").agg(
    total_enq_amt=("enq_amt", "sum"),
    mean_enq_amt=("enq_amt", "mean"),
    num_enquiries=("customer_no", "count"),
    mean_diff_open_enquiry_dt=("diff_open_enquiry_dt", "mean"),
    max_freq_enquiry=("enq_purpose", lambda x: x.mode()[0] if not x.mode().empty else "Unknown"),
).reset_index()
 
recency_90 = cust_enq[cust_enq["days_since_enquiry"] <= 90].groupby("customer_no").size()
recency_365 = cust_enq[cust_enq["days_since_enquiry"] <= 365].groupby("customer_no").size()
enq_agg = enq_agg.merge(recency_90.rename("count_enquiry_recency_90"), on="customer_no", how="left")
enq_agg = enq_agg.merge(recency_365.rename("count_enquiry_recency_365"), on="customer_no", how="left")
enq_agg["count_enquiry_recency_90"] = enq_agg["count_enquiry_recency_90"].fillna(0)
enq_agg["count_enquiry_recency_365"] = enq_agg["count_enquiry_recency_365"].fillna(0)
 

In [36]:
df = cust_demo.merge(account, on="customer_no", how="left")
df = df.merge(enq_agg, on="customer_no", how="left")
 
print("\nFinal merged shape:", df.shape)
print("Bad_label distribution:")



Final merged shape: (23896, 103)
Bad_label distribution:


In [37]:
print(df["Bad_label"].value_counts(dropna=False))

Bad_label
0    22892
1     1004
Name: count, dtype: int64


In [38]:
feature_cols = [c for c in df.columns if c.startswith("feature_")]
for c in feature_cols:
    df[c] = df[c].fillna(df[c].median())
 
# Customers with NO accounts/enquiries at all -> counts/totals should be 0, not "unknown"
count_like_cols = [
    "num_accounts", "num_enquiries", "count_enquiry_recency_90", "count_enquiry_recency_365",
    "total_high_credit_amt", "total_cur_balance_amt", "total_credit_limit", "total_cash_limit",
    "total_amt_past_due", "total_enq_amt",
]
for c in count_like_cols:
    if c in df.columns:
        df[c] = df[c].fillna(0)

In [39]:
ratio_mean_cols = [
    "mean_cur_balance_amt", "mean_credit_limit", "mean_cash_limit", "mean_rateofinterest",
    "ratio_currbalance_creditlimit", "total_diff_lastpaymt_opened_dt", "mean_diff_lastpaymt_opened_dt",
    "mean_enq_amt", "mean_diff_open_enquiry_dt",
]
for c in ratio_mean_cols:
    if c in df.columns:
        df[c] = df[c].fillna(df[c].median())
 
if "max_freq_enquiry" in df.columns:
    df["max_freq_enquiry"] = df["max_freq_enquiry"].fillna("Unknown")

In [40]:
df = df.dropna(subset=["Bad_label"])
 

In [41]:
if "max_freq_enquiry" in df.columns:
    df["max_freq_enquiry"] = LabelEncoder().fit_transform(df["max_freq_enquiry"].astype(str))

In [42]:
drop_cols = ["customer_no", "dt_opened", "entry_time", "Bad_label"]
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
X = X.select_dtypes(include=[np.number])  # keep numeric-only columns
y = df["Bad_label"].astype(int)
 
print("\nFinal feature matrix shape:", X.shape)


Final feature matrix shape: (23896, 99)


In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [45]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
 


C:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
C:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
C:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [46]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="auc",
)
xgb.fit(X_train, y_train)
xgb_proba = xgb.predict_proba(X_test)[:, 1]
 

In [47]:
def gini_score(y_true, y_proba):
    auc = roc_auc_score(y_true, y_proba)
    return 2 * auc - 1
 

In [49]:
print("\n=== MODEL COMPARISON ===")
print(f"XGBoost             - AUC: {roc_auc_score(y_test, xgb_proba):.4f} | Gini: {gini_score(y_test, xgb_proba)*100:.2f}")
print("Benchmark Gini (from PDF): 37.9")


=== MODEL COMPARISON ===
XGBoost             - AUC: 0.6696 | Gini: 33.93
Benchmark Gini (from PDF): 37.9


In [50]:
importance = xgb.get_booster().get_score(importance_type="gain")
importance_df = pd.DataFrame(
    [(k, v) for k, v in importance.items()], columns=["feature", "gain"]
).sort_values("gain", ascending=False)
 
print("\n=== Top 15 features by gain ===")
print(importance_df.head(15).to_string(index=False))


=== Top 15 features by gain ===
                       feature       gain
                     feature_7 101.727188
     count_enquiry_recency_365  94.029037
              total_cash_limit  84.308052
                    feature_52  74.107506
                    feature_41  71.197464
 ratio_currbalance_creditlimit  69.873909
            total_credit_limit  69.760727
          mean_cur_balance_amt  68.231766
                  num_accounts  64.980125
            total_amt_past_due  64.141212
                    feature_55  64.040764
                  mean_enq_amt  61.461372
total_diff_lastpaymt_opened_dt  61.293358
                     feature_3  60.257648
                    feature_64  59.830814
